In [1]:
import numpy as np
import matplotlib.pyplot as plt
import scienceplots
import os
import pandas as pd
plt.style.use(['science','notebook','grid'])
from datetime import datetime as dt

---
---

#                READ DATA FROM STORAGE

---
---

In [2]:
# read a csv file
filename = "results_proper.csv"
df = pd.read_csv(f"initial_output/simulation_results/{filename}")

In [3]:
df

,method,E0,n_hvl_x,n_hvl_y,n_hvl_z,Nsim,Srep_index,force_first_interaction,path_extension_factor,weight_min,...,N_backscattered,E_backscattered,N_leakage,E_leakage,N_absorbed,E_absorbed,buildup_factor,N_stps_per_sec,N_stps_per_Nsim,Nsim_per_sec
0,combined,1,2,10,10,10000,0,True,2.0,0.01,...,310,5.420780,2,0.010555,6888,6888,1.324775,5558.924944,3.2740,1697.900105
1,combined,1,2,10,10,10000,1,True,2.0,0.01,...,303,6.182999,1,0.000029,6560,6560,1.345877,5417.877771,3.2267,1679.077005
2,combined,1,2,10,10,10000,2,True,2.0,0.01,...,288,5.344354,0,0.000000,6846,6846,1.345221,5209.414926,3.2536,1601.123348
3,combined,1,2,10,10,10000,3,True,2.0,0.01,...,300,5.181340,0,0.000000,6860,6860,1.340332,5569.500709,3.2530,1712.112115
4,combined,1,2,10,10,10000,4,True,2.0,0.01,...,298,5.742958,0,0.000000,6850,6850,1.338017,5051.726513,3.2606,1549.324208
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
175,forcing,5,6,10,10,10000,0,True,NaN,0.01,...,1196,396.088992,9,4.779778,-14441,-72205,1.800412,4271.412991,6.8624,622.437193
176,forcing,5,6,10,10,10000,1,True,NaN,0.01,...,1137,384.787281,9,5.336685,-14629,-73145,1.703024,4431.035896,6.8873,643.363277
177,forcing,5,6,10,10,10000,2,True,NaN,0.01,...,1174,387.820798,4,1.087662,-14440,-72200,1.719987,3345.692392,6.8776,486.462195
178,forcing,5,6,10,10,10000,3,True,NaN,0.01,...,1151,384.153147,5,3.103849,-14506,-72530,1.900376,3135.878148,6.8700,456.459701


In [4]:
df.keys()

Index(['method', 'E0', 'n_hvl_x', 'n_hvl_y', 'n_hvl_z', 'Nsim', 'Srep_index',
       'force_first_interaction', 'path_extension_factor', 'weight_min',
       'survival_probability', 'simulation_time_sec', 'N_steps', 'N_out_total',
       'E_out_total', 'N_out_primaries', 'E_out_primaries',
       'N_out_secondaries', 'E_out_secondaries', 'N_backscattered',
       'E_backscattered', 'N_leakage', 'E_leakage', 'N_absorbed', 'E_absorbed',
       'buildup_factor', 'N_stps_per_sec', 'N_stps_per_Nsim', 'Nsim_per_sec'],
      dtype='object')

In [5]:
# cleanup - remove unnecessary columns from the dataframe - based on keys names
columns_to_drop = [
    "aa", #! this one is not actual key, errors are ignored using errors="ignore"
    "box_shape_hvl",
    #"Nsim",
    "force_first_interaction",
    "path_extension_factor",
    "weight_min",
    "survival_probability",
    "box_size_cm",
    "verbosity",
    "Emin_terminate",
    "N_absorbed",
    "E_absorbed",
    "N_stps_per_sec",
    "N_stps_per_Nsim",
    "Nsim_per_sec",

    # additional
    "n_hvl_y",
    "n_hvl_z",
    "N_out_total",
    "E_out_total",
    "N_out_primaries",
    "E_out_primaries",
    #"N_out_secondaries",
    "E_out_secondaries",
    "N_backscattered",
    "E_backscattered",
    "N_leakage",
    "E_leakage",
]
df = df.drop(columns=columns_to_drop, errors="ignore")

In [19]:
# calculate main statistics for buildup factor for each method, E0 and N_hvl
df.groupby(["method", "E0", "n_hvl_x"])["buildup_factor"].agg(["mean", "std"])
# add mean time per simulation for each method, E0 and N_hvl
df.groupby(["method", "E0", "n_hvl_x"])["simulation_time_sec"].agg(["mean"])

mean
method   E0 n_hvl_x          
combined 2  4        2.793316
            6        3.172516
         4  4        3.132262
            6        4.503148
pdf      2  4        1.100287
            6        1.264569
         4  4        1.343571
            6        1.438422

In [6]:
grouped = df.groupby(["method", "E0", "n_hvl_x"])

stats = grouped.agg({
    "buildup_factor": ["mean", "std"],
    "simulation_time_sec": "mean"
})

stats["FOM"] = 1 / (stats[("buildup_factor","std")]**2 * stats[("simulation_time_sec","mean")])
stats

buildup_factor           simulation_time_sec          FOM
                              mean       std                mean             
method   E0 n_hvl_x                                                          
buildup  1  2             1.352992  0.010824            2.223213  3838.965402
            4             1.632174  0.077156            2.979797    56.372703
            6             1.870789  0.049421            3.129575   130.826393
         2  2             1.403870  0.021889            3.239564   644.262270
            4             1.819970  0.045933            4.372817   108.390311
            6             2.095462  0.061374            4.844255    54.803152
         5  2             1.231788  0.011859            4.074237  1745.120595
            4             1.520956  0.066258            5.187911    43.907329
            6             1.854958  0.074506            5.556819    32.418117
combined 1  2             1.338844  0.008529            6.077212  2262.048926
            4             1.512592  0.007007            7.072511  2879.418350
            6             1.625688  0.027657            7.649672   170.896874
         2  2             1.478471  0.006000            7.233362  3840.172400
            4             1.763904  0.027434            8.490087   156.493657
            6             2.036600  0.006809            9.221240  2339.393168
         5  2             1.323155  0.007691            9.840641  1717.904204
            4             1.576516  0.023965           13.926208   125.032176
            6             1.794663  0.019944           12.465132   201.679277
forcing  1  2             1.359196  0.004002            7.035409  8873.492674
            4             1.646294  0.012922            8.527381   702.316347
            6             1.906337  0.064274            8.987928    26.931827
         2  2             1.408023  0.003932            8.773723  7372.171072
            4             1.794893  0.019325           12.377656   216.329077
            6             2.129266  0.074779           12.779529    13.993476
         5  2             1.245528  0.006610           12.608971  1815.065825
            4             1.489387  0.018930           14.693082   189.935308
            6             1.780498  0.078157           18.835737     8.691179
pdf      1  2             1.364177  0.008729            1.601780  8193.697194
            4             1.651781  0.037940            2.311682   300.516639
            6             1.871333  0.058586            2.827461   103.042266
         2  2             1.393655  0.011763            2.346567  3080.009527
            4             1.770124  0.019352            3.322846   803.618758
            6             2.140503  0.070962            4.154517    47.800519
         5  2             1.242651  0.014488            2.760283  1725.965231
            4             1.486125  0.034809            4.165702   198.122869
            6             1.766072  0.060839            5.058452    53.410213